# Multi-head Model: Predict Price + Risk Together (Real Data)

## Dataset: California Housing (scikit-learn)

### What you’ll build
A single shared backbone that predicts:
- **price** (regression head)
- **risk / expensive** (classification head) — e.g., top quartile of price

This is common in production: one model, multiple related outputs.

---


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score

import tensorflow as tf

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


/Users/omarcharif/Work/repos/bootcamp-module-assessment/.venv/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
def load_dataset() -> pd.DataFrame:
    """Load California housing and rename target to `price`."""
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    df.rename(columns={"MedHouseVal": "price"}, inplace=True)
    return df

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add simple engineered features."""
    out = df.copy()
    eps = 1e-6
    out["RoomsPerOccup"] = out["AveRooms"] / (out["AveOccup"] + eps)
    out["BedrmsPerRoom"] = out["AveBedrms"] / (out["AveRooms"] + eps)
    out["PopPerOccup"] = out["Population"] / (out["AveOccup"] + eps)
    return out

def make_preprocessor(cols: list[str]) -> ColumnTransformer:
    """Numeric preprocessing: impute + scale."""
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), cols)
    ])

def to_dense(x: Any) -> np.ndarray:
    """Convert sparse matrix to dense if needed."""
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

df = add_features(load_dataset())
df.head()


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price,RoomsPerOccup,BedrmsPerRoom,PopPerOccup
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,2.732918,0.146591,125.999951
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,2.956683,0.155797,1137.999461
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,2.957660,0.129516,176.999937
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,2.283153,0.184458,218.999914
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,2.879645,0.172096,258.999881


## Create both targets: price (regression) + expensive (classification)

In [7]:
def make_expensive_label(y_train: np.ndarray, y: np.ndarray, percentile: float = 75.0) -> Tuple[np.ndarray, float]:
    """Create expensive label based on training percentile threshold."""
    thr = float(np.percentile(y_train, percentile))
    return (y >= thr).astype(int), thr

X = df.drop(columns=["price"])
y_price = df["price"].astype(np.float32).values

X_tr_raw, X_va_raw, y_price_tr, y_price_va = train_test_split(
    X, y_price, test_size=0.2, random_state=RANDOM_STATE
)

y_exp_tr, thr = make_expensive_label(y_price_tr, y_price_tr, percentile=75.0)
y_exp_va, _ = make_expensive_label(y_price_tr, y_price_va, percentile=75.0)

print("Expensive threshold:", thr)
print("Train positive rate:", y_exp_tr.mean(), "| Valid positive rate:", y_exp_va.mean())


Expensive threshold: 2.651249885559082
Train positive rate: 0.25 | Valid positive rate: 0.24612403100775193


## Preprocess + baseline single-task models (for comparison)

In [8]:
cols = X.columns.tolist()
pre = make_preprocessor(cols)

Xtr = to_dense(pre.fit_transform(X_tr_raw))
Xva = to_dense(pre.transform(X_va_raw))

def build_price_only(input_dim: int) -> tf.keras.Model:
    """Baseline single-task price regressor."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    out = tf.keras.layers.Dense(1, name="price")(x)
    m = tf.keras.Model(inputs, out, name="price_only")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")])
    return m

def build_risk_only(input_dim: int) -> tf.keras.Model:
    """Baseline single-task expensive classifier."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid", name="risk")(x)
    m = tf.keras.Model(inputs, out, name="risk_only")
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc"), "accuracy"])
    return m

price_only = build_price_only(Xtr.shape[1])
price_only.fit(Xtr, y_price_tr, validation_data=(Xva, y_price_va), epochs=40, batch_size=128, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_rmse", mode="min", patience=8, restore_best_weights=True)])
pred_price = price_only.predict(Xva, verbose=0).ravel()
baseline_rmse = np.sqrt(mean_squared_error(y_price_va, pred_price))

risk_only = build_risk_only(Xtr.shape[1])
risk_only.fit(Xtr, y_exp_tr, validation_data=(Xva, y_exp_va), epochs=40, batch_size=128, verbose=0,
             callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True)])
proba_risk = risk_only.predict(Xva, verbose=0).ravel()
baseline_auc = roc_auc_score(y_exp_va, proba_risk)
baseline_acc = accuracy_score(y_exp_va, (proba_risk >= 0.5).astype(int))

print("Baseline price-only RMSE:", baseline_rmse)
print("Baseline risk-only AUC:", baseline_auc, "ACC:", baseline_acc)


Baseline price-only RMSE: 0.52270550574152
Baseline risk-only AUC: 0.9509116981762241 ACC: 0.9016472868217055


## Multi-head model (shared backbone + two heads)

In [10]:
def build_multihead(input_dim: int, lr: float = 1e-3) -> tf.keras.Model:
    """Build a multi-head model: one regression head (price) and one classification head (risk)."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    shared = tf.keras.layers.Dense(64, activation="relu", name="shared_repr")(x)

    price_out = tf.keras.layers.Dense(1, name="price_output")(shared)
    risk_out = tf.keras.layers.Dense(1, activation="sigmoid", name="risk_output")(shared)

    model = tf.keras.Model(inputs, outputs={"price_output": price_out, "risk_output": risk_out}, name="multihead_model")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss={"price_output": "mse", "risk_output": "binary_crossentropy"},
        metrics={
            "price_output": [tf.keras.metrics.RootMeanSquaredError(name="rmse")],
            "risk_output": [tf.keras.metrics.AUC(name="auc"), "accuracy"],
        },
        # Loss weights can be tuned; start with 1.0 / 1.0
        loss_weights={"price_output": 1.0, "risk_output": 1.0},
    )
    return model

multi = build_multihead(Xtr.shape[1], lr=1e-3)
multi.fit(
    Xtr,
    {"price_output": y_price_tr, "risk_output": y_exp_tr},
    validation_data=(Xva, {"price_output": y_price_va, "risk_output": y_exp_va}),
    epochs=60,
    batch_size=128,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_price_output_rmse", mode="min", patience=10, restore_best_weights=True)],
)

preds = multi.predict(Xva, verbose=0)
mh_price = preds["price_output"].ravel()
mh_risk = preds["risk_output"].ravel()

mh_rmse = np.sqrt(mean_squared_error(y_price_va, mh_price))
mh_auc = roc_auc_score(y_exp_va, mh_risk)
mh_acc = accuracy_score(y_exp_va, (mh_risk >= 0.5).astype(int))

print("Multi-head RMSE:", mh_rmse)
print("Multi-head AUC:", mh_auc, "ACC:", mh_acc)


Multi-head RMSE: 0.5189552268853226
Multi-head AUC: 0.9545324297107464 ACC: 0.9060077519379846


## Summary: baseline vs multi-head

In [11]:
summary = {
    "baseline": {"price_rmse": float(baseline_rmse), "risk_auc": float(baseline_auc), "risk_acc": float(baseline_acc)},
    "multihead": {"price_rmse": float(mh_rmse), "risk_auc": float(mh_auc), "risk_acc": float(mh_acc)},
    "delta": {
        "rmse_reduction": float(baseline_rmse - mh_rmse),
        "auc_gain": float(mh_auc - baseline_auc),
        "acc_gain": float(mh_acc - baseline_acc),
    }
}
summary


{'baseline': {'price_rmse': 0.52270550574152,
  'risk_auc': 0.9509116981762241,
  'risk_acc': 0.9016472868217055},
 'multihead': {'price_rmse': 0.5189552268853226,
  'risk_auc': 0.9545324297107464,
  'risk_acc': 0.9060077519379846},
 'delta': {'rmse_reduction': 0.0037502788561973865,
  'auc_gain': 0.003620731534522248,
  'acc_gain': 0.0043604651162790775}}